# Кластеризация 

Подгрузка необходимых библиотек

In [37]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, Button, VBox, Output
import pandas as pd

Подгружаем данные 

In [38]:
df = pd.read_csv('diamonds.csv')

In [39]:
df

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...,...
53935,53936,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,53937,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,53938,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,53939,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


Удалим ненужный стоблец, повторяющий нумерацию строк 

In [40]:
df= df.drop(columns=["Unnamed: 0"])

In [41]:
df

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


Размерность данных слишком большая, количество наблюдений 53940, соответственно сократим её. Выберем 500 случайных наблюдений.

In [42]:
df = df.sample(n = 1500, random_state = 667)

In [43]:
df

,carat,cut,color,clarity,depth,table,price,x,y,z
34708,0.39,Ideal,H,SI1,61.8,55.0,874,4.68,4.71,2.90
26771,2.06,Very Good,F,SI2,61.5,56.0,16603,8.15,8.18,5.02
52444,0.53,Premium,E,VVS1,60.8,59.0,2514,5.23,5.17,3.16
50912,0.72,Premium,H,VS2,61.1,58.0,2318,5.78,5.74,3.52
39892,0.42,Ideal,F,VS1,61.4,57.0,1103,4.83,4.78,2.95
...,...,...,...,...,...,...,...,...,...,...
10431,1.06,Very Good,H,SI1,62.8,58.0,4785,6.47,6.49,4.07
49395,0.75,Ideal,J,VS2,61.2,56.0,2104,5.86,5.90,3.60
35393,0.41,Very Good,G,VS1,61.7,57.0,899,4.75,4.78,2.94
49885,0.63,Premium,H,VVS1,61.3,61.0,2177,5.49,5.44,3.35


Проверим данные на наличие пропущенных значений

In [44]:
missing_counts = df.isnull().sum()
print(missing_counts)

carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64


In [45]:
print("\n2.3 Анализ типов данных")
print(df.dtypes)


2.3 Анализ типов данных
carat      float64
cut         object
color       object
clarity     object
depth      float64
table      float64
price        int64
x          float64
y          float64
z          float64
dtype: object


In [46]:
print("\n2.5 Анализ категориальных признаков")
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\nРаспределение значений в столбце '{col}':")
    value_counts = df[col].value_counts()
    print(value_counts)
    print(f"Количество уникальных значений: {df[col].nunique()}")


2.5 Анализ категориальных признаков

Распределение значений в столбце 'cut':
cut
Ideal        582
Very Good    372
Premium      354
Good         150
Fair          42
Name: count, dtype: int64
Количество уникальных значений: 5

Распределение значений в столбце 'color':
color
G    326
E    266
F    260
H    222
D    201
I    144
J     81
Name: count, dtype: int64
Количество уникальных значений: 7

Распределение значений в столбце 'clarity':
clarity
SI1     360
VS2     344
SI2     252
VS1     232
VVS2    149
VVS1     92
IF       57
I1       14
Name: count, dtype: int64
Количество уникальных значений: 8


Проведем кодировку категориальных признаков 

In [47]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [48]:
from sklearn.preprocessing import LabelEncoder
# Create a label encoder object
label_encoder = LabelEncoder()

df = df.copy()
# Transform the 'categories' column
df['cut'] = label_encoder.fit_transform(df['cut'].values)
df['color'] = label_encoder.fit_transform(df['color'].values)

df['clarity'] = label_encoder.fit_transform(df['clarity'].values)

df

,carat,cut,color,clarity,depth,table,price,x,y,z
34708,0.39,2,4,2,61.8,55.0,874,4.68,4.71,2.90
26771,2.06,4,2,3,61.5,56.0,16603,8.15,8.18,5.02
52444,0.53,3,1,6,60.8,59.0,2514,5.23,5.17,3.16
50912,0.72,3,4,5,61.1,58.0,2318,5.78,5.74,3.52
39892,0.42,2,2,4,61.4,57.0,1103,4.83,4.78,2.95
...,...,...,...,...,...,...,...,...,...,...
10431,1.06,4,4,2,62.8,58.0,4785,6.47,6.49,4.07
49395,0.75,2,6,5,61.2,56.0,2104,5.86,5.90,3.60
35393,0.41,4,3,4,61.7,57.0,899,4.75,4.78,2.94
49885,0.63,3,4,6,61.3,61.0,2177,5.49,5.44,3.35


Подготовим данные к кластеризации, стандартизируем данные

In [49]:
# Подготовка данных для кластеризации
X = df.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [50]:
# Perform PCA to reduce to 2 dimensions for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

In [51]:
from sklearn.cluster import DBSCAN

In [58]:
from ipywidgets import IntSlider, FloatSlider, Button, VBox, Output, HBox, Label

In [61]:
def plot_dbscan_clusters(eps, min_samples):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_scaled)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    plt.figure(figsize=(10, 6))
    
    unique_labels = set(labels)
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    
    for i, label in enumerate(sorted(unique_labels)):
        if label == -1:
            noise_points = X_pca[labels == label]
            plt.scatter(noise_points[:, 0], noise_points[:, 1], 
                       c='gray', s=30, alpha=0.5, label=f'Noise ({n_noise} points)')
        else:
            cluster_points = X_pca[labels == label]
            plt.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                       c=colors[i], s=50, alpha=0.7, label=f'Cluster {label}')
    
    plt.title(f'DBSCAN Clustering (eps={eps}, min_samples={min_samples})')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend(title='Clusters', loc='best')
    plt.grid(True)
    plt.show()

eps_slider = FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description='Eps:')
min_samples_slider = IntSlider(value=5, min=2, max=20, step=1, description='Min Samples:')
update_button = Button(description='Update Visualization')

output = Output()

def on_button_click(b):
    with output:
        output.clear_output(wait=True)
        plot_dbscan_clusters(eps_slider.value, min_samples_slider.value)

update_button.on_click(on_button_click)

display(VBox([eps_slider, min_samples_slider, update_button, output]))

with output:
    plot_dbscan_clusters(eps_slider.value, min_samples_slider.value)

# Регрессия

In [69]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("framingham (1).csv")

target_col = "TenYearCHD"

feature_candidates = [
    "male", "age", "education", "currentSmoker", "cigsPerDay",
    "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes",
    "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"
]

# Оставляем только те признаки, которые реально есть в файле
feature_candidates = [c for c in feature_candidates if c in df.columns]

# Удаляем строки с пропусками по признакам/таргету (для простоты)
df = df[feature_candidates + [target_col]].dropna()

X_all = df[feature_candidates]
y = df[target_col].astype(int)

# 2. Виджет выбора признаков
feature_select = widgets.SelectMultiple(
    options=feature_candidates,
    value=tuple(feature_candidates),   # по умолчанию все признаки
    description="Features",
    disabled=False,
    layout=widgets.Layout(width='50%', height='200px')
)

# Кнопка обновления
update_button = widgets.Button(
    description="Update Model",
    button_style="success"
)

# Output для результатов
output = widgets.Output()

def update_model(button):
    with output:
        clear_output()
        
        selected_features = list(feature_select.value)
        if len(selected_features) == 0:
            print("Please select at least one feature.")
            return
        
        X = X_all[selected_features]
        
        # 3. Train / test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )
        
        # 4. Масштабирование признаков
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # 5. Логистическая регрессия (scikit-learn)
        clf = LogisticRegression(max_iter=1000, solver="lbfgs")
        clf.fit(X_train_scaled, y_train)
        
        # Предсказания
        y_pred = clf.predict(X_test_scaled)
        y_proba = clf.predict_proba(X_test_scaled)[:, 1]
        
        # 6. Метрики качества
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_proba)
        
        print("Selected features:", selected_features)
        print(f"Accuracy : {acc:.3f}")
        print(f"Precision: {prec:.3f}")
        print(f"Recall   : {rec:.3f}")
        print(f"F1-score : {f1:.3f}")
        print(f"ROC AUC  : {auc:.3f}")
        
        # 7. ROC-кривая
        fpr, tpr, thresholds = roc_curve(y_test, y_proba)
        
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
        plt.plot([0, 1], [0, 1], 'k--', label="Random")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC curve")
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.show()

# Привязка обработчика
update_button.on_click(update_model)

# Отображение виджетов
display(widgets.VBox([feature_select, update_button, output]))

# Первичный запуск
with output:
    update_model(None)